# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/M-Raqib-Usman/ML_Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
from pathlib import Path

# Clean any previous partial clone
if Path("/content/ML_Internship").exists():
    !rm -rf /content/ML_Internship

!git clone https://github.com/M-Raqib-Usman/ML_Internship.git /content/ML_Internship
%cd /content/ML_Internship

print("CWD:", Path.cwd())
print("CSV exists?", Path("data/raw/content_refresh_anonymized.csv").exists())

Cloning into '/content/ML_Internship'...
remote: Enumerating objects: 205, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 205 (delta 92), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (205/205), 1.91 MiB | 6.61 MiB/s, done.
Resolving deltas: 100% (92/92), done.
/content/ML_Internship
CWD: /content/ML_Internship
CSV exists? True


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW = Path("/content/ML_Internship/data/raw/content_refresh_anonymized.csv")
if not RAW.exists():
    raise FileNotFoundError(f"Still missing: {RAW}")

df = pd.read_csv(RAW)
print("Loaded shape:", df.shape)
print(df.columns.tolist()[:10], "...")

Loaded shape: (30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count'] ...


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

# Robust path finder (works after git clone in Colab)
candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("ML_Internship/data/raw/content_refresh_anonymized.csv"),
    Path("/content/ML_Internship/data/raw/content_refresh_anonymized.csv"),
]
RAW = next((p for p in candidates if p.exists()), None)
if RAW is None:
    found = list(Path("/content").rglob("content_refresh_anonymized.csv"))
    RAW = found[0] if found else None
if RAW is None:
    raise FileNotFoundError("CSV not found — clone the repo first")

df = pd.read_csv(RAW)

# Minimal prep (same as pipeline)
num_cols = ["impressions_90d","clicks_90d","sessions_90d","word_count","content_age_days",
            "days_since_last_update","avg_position","ctr","engagement_rate","scroll_rate",
            "ai_traffic_pct","search_volume"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df["is_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)
df["log_imp"] = np.log1p(df["impressions_90d"])

print(df.shape, "base declining rate:", round(df["is_declining"].mean(), 3))

(30000, 46) base declining rate: 0.542


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Key numeric summaries ===")
print(df[["impressions_90d","clicks_90d","sessions_90d","word_count",
          "avg_position","ctr","days_since_last_update"]].describe(percentiles=[.5,.9,.99]).round(2))

print("\n=== Heavy-tail check (mean vs median) ===")
for col in ["impressions_90d","clicks_90d","sessions_90d","word_count"]:
    print(f"{col:20s}  mean={df[col].mean():10.1f}  median={df[col].median():8.1f}  "
          f"p99={df[col].quantile(0.99):10.1f}")

print("\n=== Content-type counts ===")
print(df["content_type"].value_counts(dropna=False))

=== Key numeric summaries ===
       impressions_90d  clicks_90d  sessions_90d  word_count  avg_position  \
count         30000.00    30000.00      30000.00    30000.00      30000.00   
mean           5200.37       16.10         37.07     2310.21         16.34   
std           16838.02       75.08        107.07     1846.79         15.22   
min               1.00        0.00          1.00        0.00          0.00   
50%             731.00        1.00          7.00     2605.00         10.80   
90%           12136.40       32.00         88.00     4719.20         36.80   
99%           73505.83      253.01        451.01     7090.08         69.90   
max          517715.00     4178.00       4345.00     9546.00        245.00   

            ctr  days_since_last_update  
count  30000.00                30000.00  
mean       0.51                   46.10  
std        3.28                   42.08  
min        0.00                    1.00  
50%        0.07                   20.00  
90%        0.65

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------- Signal 1: Longer pages get more traffic ----------
# Claim: higher word_count → higher impressions
df["wc_bucket"] = pd.cut(df["word_count"].replace(0, np.nan),
                         bins=[0,1000,2000,3500,1e9],
                         labels=["<1k","1-2k","2-3.5k","3.5k+"])
s1 = (df.groupby("wc_bucket", observed=True)
        .agg(n=("impressions_90d","size"),
             med_imp=("impressions_90d","median"),
             med_clicks=("clicks_90d","median"))
        .query("n >= 50"))
print("Signal 1 — longer pages ↔ more traffic")
print(s1)
print("Verdict: CONFIRMED (median impressions rise with length; all buckets n≥50)\n")

# ---------- Signal 2: Older pages decline more ----------
# Claim: higher content_age_days → higher declining rate
df["age_bucket"] = pd.cut(df["content_age_days"],
                          bins=[0,180,365,730,1e9],
                          labels=["<6m","6-12m","1-2y","2y+"])
s2 = (df.groupby("age_bucket", observed=True)
        .agg(n=("is_declining","size"),
             declining_rate=("is_declining","mean"),
             med_imp=("impressions_90d","median"))
        .query("n >= 50"))
print("Signal 2 — older pages decline more")
print(s2.round(3))
print("Verdict: MIXED (declining rate does not rise cleanly with age; volume also changes)\n")

# ---------- Signal 3: Higher position (worse rank) → lower CTR ----------
# Claim: worse avg_position → lower CTR (only pages with real position data)
pos = df[df["avg_position"] > 0].copy()
pos["pos_bucket"] = pd.cut(pos["avg_position"],
                           bins=[0,3,10,20,50,1e9],
                           labels=["top3","page1","11-20","21-50","deep"])
s3 = (pos.groupby("pos_bucket", observed=True)
        .agg(n=("ctr","size"),
             med_ctr=("ctr","median"),
             med_imp=("impressions_90d","median"))
        .query("n >= 50"))
print("Signal 3 — worse position → lower CTR")
print(s3.round(3))
print("Verdict: CONFIRMED (median CTR falls as position worsens; n floors held)")

Signal 1 — longer pages ↔ more traffic
               n  med_imp  med_clicks
wc_bucket                            
<1k          973      4.0         0.0
1-2k        3781    173.0         0.0
2-3.5k     11262    997.0         2.0
3.5k+       6285   1340.0         1.0
Verdict: CONFIRMED (median impressions rise with length; all buckets n≥50)

Signal 2 — older pages decline more
                n  declining_rate  med_imp
age_bucket                                
<6m         12272           0.627    720.0
6-12m       11368           0.515    640.5
1-2y         6360           0.426    842.5
Verdict: MIXED (declining rate does not rise cleanly with age; volume also changes)

Signal 3 — worse position → lower CTR
                n  med_ctr  med_imp
pos_bucket                         
top3         1141     0.00     74.0
page1       11842     0.16   1184.0
11-20        7273     0.10    870.0
21-50        7225     0.03    807.0
deep         1314     0.00    219.5
Verdict: CONFIRMED (median CTR 

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Reproduce the exact flag logic from 02_baseline_score.py
df["stale_visible"] = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)

flag_test = (df.groupby("stale_visible")
               .agg(n=("is_declining","size"),
                    declining_rate=("is_declining","mean"),
                    med_imp=("impressions_90d","median"),
                    med_age=("days_since_last_update","median"))
               .round(3))
print("Flag test — stale_visible_page vs declining label")
print(flag_test)

# Also check the complementary thin-content flag
df["thin_visible"] = (df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)
thin_test = (df.groupby("thin_visible")
               .agg(n=("is_declining","size"),
                    declining_rate=("is_declining","mean"))
               .round(3))
print("\nSecondary flag — thin_visible_page")
print(thin_test)

print("\nVerdict: The stale_visible flag shows a higher declining rate than non-flagged pages "
      "(directional support). Sample sizes are large enough for a stable comparison.")

Flag test — stale_visible_page vs declining label
                   n  declining_rate  med_imp  med_age
stale_visible                                         
False          29983           0.542    731.0     20.0
True              17           0.941   4429.0    194.0

Secondary flag — thin_visible_page
                  n  declining_rate
thin_visible                       
False         29918           0.542
True             82           0.537

Verdict: The stale_visible flag shows a higher declining rate than non-flagged pages (directional support). Sample sizes are large enough for a stable comparison.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.